In [1]:
# Making a list of equivalent names for medicines.

In [1]:
import pandas as pd, re, json, unicodedata
from collections import defaultdict

# ---------------------------------------------------------------------------
# 1. Normalise a token (same rule everywhere)
# ---------------------------------------------------------------------------
_RX_QUOTES = re.compile(r'"[^"]*"')

def norm(token: str) -> str:
    if pd.isna(token):
        return ''
    token = unicodedata.normalize('NFKC', str(token))   # Unicode clean-up
    token = _RX_QUOTES.sub('', token)                   # drop “…”
    token = re.sub(r'\s+', ' ', token).strip().lower()  # collapse ws + lower
    return token


# ---------------------------------------------------------------------------
# 2. Load the two Excel sheets (only needed columns)
# ---------------------------------------------------------------------------
discont = pd.read_excel(
    'Afregistrerede_Laegemidler.xls',
    sheet_name='AfregLægemidlerSidsteÅr',
    usecols=['Lægemiddel', 'AktiveSubstanser']
)

current = pd.read_excel(
    'ListeOverGodkendteLaegemidler.xlsx',
    sheet_name='Godkendte Lægemidler',
    usecols=['Navn', 'AktiveSubstanser']
)


# ---------------------------------------------------------------------------
# 3. Union–find helpers  (for merging overlapping synonym groups)
# ---------------------------------------------------------------------------
parent = {}

def find(x):
    parent.setdefault(x, x)
    if parent[x] != x:
        parent[x] = find(parent[x])     # path compression
    return parent[x]

def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[rb] = ra


# ---------------------------------------------------------------------------
# 4. Feed every row: union(synonym , generic)
# ---------------------------------------------------------------------------
def add_row(syn_raw, gen_raw):
    syn = norm(syn_raw)
    gen = norm(gen_raw)
    if not syn or not gen:
        return
    union(syn, gen)     # link the two names
    # also make sure each of them exists in parent dict
    parent.setdefault(syn, syn)
    parent.setdefault(gen, gen)

for s, g in zip(discont['Lægemiddel'], discont['AktiveSubstanser']):
    add_row(s, g)

for s, g in zip(current['Navn'], current['AktiveSubstanser']):
    add_row(s, g)


# ---------------------------------------------------------------------------
# 5. Collect final connected components  →  list-of-lists
# ---------------------------------------------------------------------------
groups_dict = defaultdict(list)
for name in parent.keys():
    groups_dict[find(name)].append(name)

equivalence_groups = [sorted(lst) for lst in groups_dict.values()]
equivalence_groups.sort(key=lambda lst: (len(lst[0]), lst))   # optional


# ---------------------------------------------------------------------------
# 6. Write JSON
# ---------------------------------------------------------------------------
with open('equivalent_names.json', 'w', encoding='utf-8') as fh:
    json.dump(equivalence_groups, fh, ensure_ascii=False, indent=2)

print(f"Created {len(equivalence_groups)} groups in equivalent_names.json")

c:\Users\danam\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Created 2594 groups in equivalent_names.json
